# Da She's Voice Engine — Colab GPU
Uses Python 3.10 via conda for TTS compatibility.

In [ ]:
# 1. Install Python 3.10 + TTS via conda
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /usr/local -f > /dev/null 2>&1
!conda install -y python=3.10 -c conda-forge -q 2>&1 | tail -2
!python3.10 -m pip install -q TTS flask flask-cors 2>&1 | tail -3
print("Python 3.10 + TTS ready")

In [ ]:
# 2. Download speaker and start server (runs with Python 3.10)
import subprocess, os
import requests

r = requests.get("https://qwert.crousia.com/speaker.wav", timeout=30)
with open("speaker.wav", "wb") as f: f.write(r.content)
print(f"Speaker: {len(r.content)} bytes")

# Write server script
server_code = '''
from flask import Flask, request, send_file
from TTS.api import TTS
import torch, tempfile, os, threading

has_gpu = torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0) if has_gpu else 'N/A'}")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=has_gpu)

app = Flask(__name__)
@app.route(\"/health\")
def health(): return {\"status\": \"ok\", \"gpu\": has_gpu}

@app.route(\"/synthesize\", methods=[\"POST\"])
def synth():
    text = request.get_json().get(\"text\", \"\")
    fd, path = tempfile.mkstemp(suffix=\".wav\"); os.close(fd)
    tts.tts_to_file(text=text, file_path=path, speaker_wav=\"speaker.wav\", language=\"en\")
    return send_file(path, mimetype=\"audio/wav\")

threading.Thread(target=lambda: app.run(host=\"0.0.0.0\", port=5000, debug=False), daemon=True).start()
print(\"Server on :5000\")
while True: import time; time.sleep(60)
'''

proc = subprocess.Popen(["/usr/local/bin/python3.10", "-c", server_code],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("Server process started")

In [ ]:
# 3. Expose via Cloudflare Tunnel
import subprocess, time, re

!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

proc2 = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
for _ in range(30):
    time.sleep(1)
    out = proc2.stdout.read(4096) if proc2.stdout else ""
    m = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', out)
    if m:
        url = m.group(0)
        break

print("=" * 60)
if url:
    print(f"COLAB GPU ENDPOINT: {url}")
    print("=" * 60)
else:
    print("Tunnel output:")
    print(out[-500:] if out else "(none)")

while True: time.sleep(60)